## Set Up

In [37]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [38]:
url_1 = 'https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/data/processed/sf_business_demographics_nhood.csv'
combined = pd.read_csv(url_1, index_col='neighborhood')

In [42]:
url_2 = 'https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/data/processed/neighborhood_table_with_years.csv'
neighborhood_table = pd.read_csv(url_2)

In [43]:
url_3 = 'https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/data/processed/netchange_yearly.csv'
netchange_yearly = pd.read_csv(url_3)

In [44]:
if 'neighborhood' in combined.columns:
    combined = combined.drop(columns='neighborhood')

## San Francisco Business Openings vs Closings (2016–2025)

In [47]:
import plotly.express as px

# Plotting
fig = px.line(
    netchange_yearly,
    x="year",
    y="count",
    color="status",
    markers=True
)

# Adding labels
fig.update_layout(
    title="San Francisco Business Openings vs Closings (2016–2025)",
    xaxis_title="Year",
    yaxis_title="Number of Businesses",
    legend_title="Status"
)

fig.update_xaxes(dtick=1)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray"
)

fig.show()


## COVID Impact vs Recovery by Neighborhood

In [ ]:
import plotly.graph_objects as go

df = combined.copy()

neighborhoods = df.index.tolist()

base_color = "lightgray"
highlight_color = "red"

x_mean = df["closure_rate"].mean()
y_mean = df["recovery_rate"].mean()

x_min, x_max = df["closure_rate"].min(), df["closure_rate"].max()
y_min, y_max = df["recovery_rate"].min(), df["recovery_rate"].max()

x_pad = (x_max - x_min) * 0.08
y_pad = (y_max - y_min) * 0.08

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["closure_rate"],
        y=df["recovery_rate"],
        mode="markers",
        text=df.index,
        customdata=df[["closures_2020_21", "openings_2022_25"]].values,
        hovertemplate=
            "<b>%{text}</b><br>" +
            "Closure rate: %{x:.3f}<br>" +
            "Recovery rate: %{y:.3f}<br>" +
            "Total closures: %{customdata[0]:.0f}<br>" +
            "Total openings: %{customdata[1]:.0f}<br>" +
            "<extra></extra>",
        marker=dict(color=base_color, size=10),
    )
)

# Quadrant lines
fig.add_shape(type="line", x0=x_mean, x1=x_mean, y0=y_min, y1=y_max,
              line=dict(dash="dash", color="black"))
fig.add_shape(type="line", x0=x_min, x1=x_max, y0=y_mean, y1=y_mean,
              line=dict(dash="dash", color="black"))

# Quadrant labels
quadrant_labels = [
    dict(x=x_max - x_pad, y=y_max - y_pad, text="High closure<br>High recovery"),
    dict(x=x_min + x_pad, y=y_max - y_pad, text="Low closure<br>High recovery"),
    dict(x=x_min + x_pad, y=y_min + y_pad, text="Low closure<br>Low recovery"),
    dict(x=x_max - x_pad, y=y_min + y_pad, text="High closure<br>Low recovery"),
]
for q in quadrant_labels:
    fig.add_annotation(
        x=q["x"], y=q["y"],
        text=q["text"],
        showarrow=False,
        font=dict(size=10, color="gray"),
        align="center",
        bgcolor="rgba(255,255,255,0.6)",
        borderpad=3
    )

# Dropdown
buttons = [
    dict(
        label="All",
        method="update",
        args=[
            {"marker": {"color": [base_color] * len(neighborhoods), "size": 10}},
            {"title": "COVID Impact vs Recovery"}
        ],
    )
]

for n in neighborhoods:
    colors = [highlight_color if name == n else base_color for name in neighborhoods]
    sizes  = [14 if name == n else 10 for name in neighborhoods]
    buttons.append(
        dict(
            label=n,
            method="update",
            args=[
                {"marker": {"color": colors, "size": sizes}},
                {"title": f"Highlight: {n}"}
            ],
        )
    )

fig.update_layout(
    title="COVID Impact vs Recovery",
    xaxis_title="Closure Rate (2020–2021)",
    yaxis_title="Recovery Rate (2022–2025)",
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True)]
)

fig.show()


## Race/Ethnicity and Median Income by Neighborhood

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, Output, VBox

out = Output()

def plot_neighborhood(neighborhood):
    row = combined.loc[neighborhood]  # use .loc since neighborhood is the index

    race_cols   = ["pct_white", "pct_black", "pct_asian", "pct_latina_o", "pct_other"]
    race_labels = ["White (NH)", "Black (NH)", "Asian & Pacific Islander (NH)", "Hispanic/Latino", "Other (NH)"]
    values = [row[c] * 100 for c in race_cols]
    income = row["median_income"]
    income_str = f"${income:,.0f}" if pd.notna(income) else "No data"

    with out:
        out.clear_output(wait=True)

        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]})
        fig.suptitle("Race/Ethnicity by Neighborhood", fontsize=13, fontweight="bold")
        ax.set_title(neighborhood, fontsize=11, color="gray", pad=8)

        bars = ax.bar(race_labels, values, color="#378ADD", width=0.5)
        ax.set_ylabel("% of population")
        ax.set_ylim(0, max(values) + 10)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=9)
        ax.tick_params(axis='x', labelsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax2.axis("off")
        ax2.text(0.5, 0.6, "Median\nhousehold\nincome", ha="center", va="center",
                 fontsize=11, color="gray", transform=ax2.transAxes, linespacing=1.6)
        ax2.text(0.5, 0.35, income_str, ha="center", va="center",
                 fontsize=18, fontweight="bold", color="#378ADD", transform=ax2.transAxes)

        plt.tight_layout()
        plt.show()

dropdown = Dropdown(options=sorted(combined.index.tolist()), description="Neighborhood:")
dropdown.observe(lambda change: plot_neighborhood(change["new"]), names="value")

plot_neighborhood(dropdown.value)
display(VBox([dropdown, out]))

## Business Resilience by Neighborhood Demographics

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Resilience vs Median Income", "Resilience vs % People of Color"),
    horizontal_spacing=0.12
)

hover_income = (
    "<b>%{text}</b><br>"
    "Resilience: %{y:.3f}<br>"
    "Median Income: $%{x:,.0f}<br>"
    "<extra></extra>"
)

hover_poc = (
    "<b>%{text}</b><br>"
    "Resilience: %{y:.3f}<br>"
    "% POC: %{x:.1%}<br>"
    "<extra></extra>"
)

fig.add_trace(go.Scatter(
    x=combined['median_income'],
    y=combined['resilience'],
    mode='markers+text',
    text=combined.index,
    textposition='top center',
    textfont=dict(size=7, color='gray'),
    hovertemplate=hover_income,
    marker=dict(
        color=combined['median_income'],
        colorscale='RdBu',
        size=10,
        showscale=True,
        colorbar=dict(title='Median Income', x=0.44),
        line=dict(width=1, color='white')
    ),
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=combined['pct_poc'],
    y=combined['resilience'],
    mode='markers+text',
    text=combined.index,
    textposition='top center',
    textfont=dict(size=7, color='gray'),
    hovertemplate=hover_poc,
    marker=dict(
        color=combined['pct_poc'],
        colorscale='RdBu_r',
        size=10,
        showscale=True,
        colorbar=dict(title='% POC', x=1.02),
        line=dict(width=1, color='white')
    ),
    showlegend=False
), row=1, col=2)

# Zero line
for col in [1, 2]:
    fig.add_shape(
        type='line', x0=0, x1=1, y0=0, y1=0,
        xref=f"x{'' if col==1 else col} domain",
        yref=f"y{'' if col==1 else col}",
        line=dict(dash='dash', color='black', width=1),
        row=1, col=col
    )

fig.update_xaxes(title_text="Median Income", tickprefix="$", tickformat=",", col=1)
fig.update_xaxes(title_text="% People of Color", tickformat=".0%", col=2)
fig.update_yaxes(title_text="Resilience (recovery rate − closure rate)", col=1)
fig.update_yaxes(title_text="", col=2)

fig.update_layout(
    title="Business Resilience by Neighborhood Demographics",
    width=1200,
    height=560,
    plot_bgcolor="white",
)

fig.show()

In [53]:
neighborhood_table.head()

,open_2019,open_2020,open_2021,open_2022,open_2023,open_2024,open_2025,close_2019,close_2020,close_2021,close_2022,close_2023,close_2024,close_2025,median_income,pct_poc
nhood,,,,,,,,,,,,,,,,
Bayview Hunters Point,551,530,452,347,340,313,189,273,318,365,318,278,337,210,96121.000000,0.923272
Bernal Heights,260,156,188,165,157,151,69,151,165,148,142,142,140,104,163135.000000,0.587381
Castro/Upper Market,354,322,256,271,264,299,122,182,244,212,193,165,227,165,202922.857143,0.337223
Chinatown,377,232,200,264,206,198,119,252,294,218,225,230,239,167,41245.000000,0.902009
Excelsior,184,149,130,115,141,125,50,127,119,89,91,109,122,79,110043.875000,0.847659


## Pre vs Post COVID Business Trajectory by Demographic Quartile

In [54]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Build net change per year per neighborhood
years = list(range(2019, 2026))

net_change = pd.DataFrame(index=neighborhood_table.index)
for y in years:
    open_col  = f"open_{y}"
    close_col = f"close_{y}"
    if open_col in neighborhood_table.columns and close_col in neighborhood_table.columns:
        net_change[y] = neighborhood_table[open_col] - neighborhood_table[close_col]

# Pull demographics directly from neighborhood_years
net_change['median_income'] = pd.to_numeric(neighborhood_table['median_income'], errors='coerce')
net_change['pct_poc'] = neighborhood_table['pct_poc']

# Assign quartiles
net_change['income_quartile'] = pd.qcut(
    net_change['median_income'], q=4,
    labels=['Q1 (lowest income)', 'Q2', 'Q3', 'Q4 (highest income)']
)
net_change['poc_quartile'] = pd.qcut(
    net_change['pct_poc'], q=4,
    labels=['Q1 (least POC)', 'Q2', 'Q3', 'Q4 (most POC)']
)

# ... rest of your code unchanged

# Average net change per year by quartile
income_trends = (
    net_change.groupby('income_quartile', observed=True)[years]
    .mean()
)
poc_trends = (
    net_change.groupby('poc_quartile', observed=True)[years]
    .mean()
)

# Colors
income_palette = ['#d6604d', '#f4a582', '#92c5de', '#2166ac']
poc_palette    = ['#2166ac', '#92c5de', '#f4a582', '#d6604d']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Avg Net Business Change by Income Quartile",
        "Avg Net Business Change by % POC Quartile"
    ),
    horizontal_spacing=0.12
)

# Chart 1: Income quartiles
for i, (q_label, row) in enumerate(income_trends.iterrows()):
    fig.add_trace(go.Scatter(
        x=years,
        y=row.values,
        mode='lines+markers',
        name=str(q_label),
        legendgroup=f"income_{q_label}",
        line=dict(color=income_palette[i], width=2.5),
        marker=dict(size=7),
        hovertemplate=f"<b>{q_label}</b><br>Year: %{{x}}<br>Avg net change: %{{y:.1f}}<extra></extra>"
    ), row=1, col=1)

# Chart 2: POC quartiles
for i, (q_label, row) in enumerate(poc_trends.iterrows()):
    fig.add_trace(go.Scatter(
        x=years,
        y=row.values,
        mode='lines+markers',
        name=str(q_label),
        legendgroup=f"poc_{q_label}",
        line=dict(color=poc_palette[i], width=2.5),
        marker=dict(size=7),
        showlegend=True,
        hovertemplate=f"<b>{q_label}</b><br>Year: %{{x}}<br>Avg net change: %{{y:.1f}}<extra></extra>"
    ), row=1, col=2)

# COVID reference line on both charts
for col in [1, 2]:
    fig.add_shape(
        type='line', x0=2020, x1=2020,
        y0=-50, y1=50,
        line=dict(dash='dash', color='black', width=1),
        row=1, col=col
    )
    fig.add_annotation(
        x=2020, y=45,
        text="COVID", showarrow=False,
        font=dict(size=9, color='black'),
        xref=f"x{'' if col==1 else col}",
        yref=f"y{'' if col==1 else col}"
    )
    fig.add_shape(
        type='line', x0=2019, x1=2025,
        y0=0, y1=0,
        line=dict(dash='dot', color='gray', width=1),
        row=1, col=col
    )

fig.update_xaxes(tickvals=years, ticktext=[str(y) for y in years])
fig.update_yaxes(title_text="Avg net change (openings − closings)", col=1)

fig.update_layout(
    title="Pre vs Post COVID Business Trajectory by Demographic Quartile",
    width=1200,
    height=520,
    plot_bgcolor="white",
    legend=dict(orientation="v", x=1.02, y=0.5)
)

fig.show()